In [6]:
import pandas as pd
import numpy as np
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import root_mean_squared_error, r2_score

In [7]:
features_map = {
    "J. Kampe": [
        "z_term_3", "z_term_2", "d_lag_2", "d_lag_3", "d_lag_4", "z_cogram", "d_lag_16", "d_lag_5",
        "d_lag_15", "d_lag_17", "d_lag_12", "d_lag_6", "d_lag_1", "z_term_4", "z_term_6", "d_lag_18",
        "d_lag_13", "d_lag_7", "d_lag_14", "z_cogram_lag_4", "z_gram_lag_4", "z_cogram_lag_5", "z_gram_lag_5", "z_gram_lag_7",
        "z_term_5", "z_gram_lag_6", "z_cogram_lag_6", "z_cogram_lag_7", "d_lag_11", "z_cogram_lag_2", "d_lag_21", "z_cogram_lag_8",
        "z_cogram_lag_11", "z_cogram_lag_3", "d_lag_8", "z_gram_lag_2", "z_gram_lag_8", "z_gram_lag_1", "z_cogram_lag_12", "d_lag_19"
    ]
}

random_forest_features = pd.read_csv("../results/random_forest_feature_selection.csv")["feature"].tolist()
correlation_features = pd.read_csv("../results/correlation_feature_selection.csv")["feature"].tolist()
gevrey_method_features = pd.read_csv("../results/gevrey_method_feature_selection.csv")["feature"].tolist()
mrmr_10_features = pd.read_csv("../results/mrmr_10_features.csv")["feature"].tolist()
mrmr_14_features = pd.read_csv("../results/mrmr_14_features.csv")["feature"].tolist()

features_map["Random Forest"] = random_forest_features
features_map["Correlation"] = correlation_features
features_map["Gevrey Method"] = gevrey_method_features
features_map["Gevrey Method (8 features)"] = gevrey_method_features[:8]    # Limiting to top 8 features
features_map["Gevrey Method (10 features)"] = gevrey_method_features[:10]  # Limiting to top 10 features
features_map["Gevrey Method (12 features)"] = gevrey_method_features[:12]  # Limiting to top 12 features
features_map["Gevrey Method (14 features)"] = gevrey_method_features[:14]  # Limiting to top 14 features
features_map["Gevrey Method (20 features)"] = gevrey_method_features[:20]  # Limiting to top 20 features
features_map["mRMR (10 features)"] = mrmr_10_features
features_map["mRMR (14 features)"] = mrmr_14_features

In [8]:
class DatasetService:
    MAX_LIMIT = 100_000

    def __init__(self, features: list[str]):
        self.X_df = pd.read_csv("../dataset/j_kampe.csv")
        self.y_df = pd.read_csv("../dataset/distances.csv")["distance"]
        self.features = features

    def get_train_test(self, limit: int = 11_000):
        if limit > self.MAX_LIMIT:
            limit = self.MAX_LIMIT

        X = self.X_df[self.features].values
        y = self.y_df.values.reshape(-1, 1)

        X = X[1_000:limit]
        y = y[1_000:limit]

        split = int(0.8 * len(X))

        return (
            X[:split],
            X[split:],
            y[:split],
            y[split:]
        )

In [9]:
def run_experiment(scaler, features_map):
    results = []

    param_grid = {
        "svr__C": [0.1, 1, 10, 100],
        "svr__epsilon": [0.001, 0.01, 0.1, 0.5],
        "svr__gamma": ["scale", 0.01, 0.1, 1.0]
    }

    for group_name, features in features_map.items():
        data = DatasetService(features)
        X_train, X_test, y_train, y_test = data.get_train_test()

        pipeline = Pipeline([
            ("scaler", scaler()),
            ("svr", SVR(kernel="rbf"))
        ])

        grid = GridSearchCV(
            pipeline,
            param_grid,
            scoring="neg_root_mean_squared_error",
            cv=3,
            n_jobs=-1
        )

        grid.fit(X_train, y_train.ravel())

        best_model = grid.best_estimator_

        y_pred = best_model.predict(X_test)

        rmse = root_mean_squared_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        results.append({
            "Group": group_name,
            "Scaler": scaler.__name__,
            "RMSE": rmse,
            "R2": r2,
            "Best C": grid.best_params_["svr__C"],
            "Best epsilon": grid.best_params_["svr__epsilon"],
            "Best gamma": grid.best_params_["svr__gamma"],
            "Features": len(features)
        })

    return pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)


In [10]:
df_minmax = run_experiment(MinMaxScaler, features_map)
print(df_minmax)
df_minmax.to_csv("../results/experiment_7_minmax_pipeline.csv", index=False)

                          Group        Scaler      RMSE        R2  Best C  \
0   Gevrey Method (20 features)  MinMaxScaler  0.023984  0.991244     100   
1   Gevrey Method (14 features)  MinMaxScaler  0.030215  0.986103      10   
2                      J. Kampe  MinMaxScaler  0.037505  0.978587     100   
3   Gevrey Method (12 features)  MinMaxScaler  0.039481  0.976272      10   
4                 Gevrey Method  MinMaxScaler  0.040743  0.974731     100   
5   Gevrey Method (10 features)  MinMaxScaler  0.061395  0.942621      10   
6                 Random Forest  MinMaxScaler  0.063755  0.938126     100   
7    Gevrey Method (8 features)  MinMaxScaler  0.072143  0.920773      10   
8                   Correlation  MinMaxScaler  0.102486  0.840114     100   
9            mRMR (14 features)  MinMaxScaler  0.110219  0.815077      10   
10           mRMR (10 features)  MinMaxScaler  0.115538  0.796797      10   

    Best epsilon Best gamma  Features  
0          0.001        1.0        

In [11]:
df_standard = run_experiment(StandardScaler, features_map)
print(df_standard)
df_standard.to_csv("../results/experiment_7_standard_pipeline.csv", index=False)

                          Group          Scaler      RMSE        R2  Best C  \
0   Gevrey Method (20 features)  StandardScaler  0.025454  0.990137     100   
1   Gevrey Method (14 features)  StandardScaler  0.028306  0.987803      10   
2                 Gevrey Method  StandardScaler  0.028631  0.987522     100   
3                      J. Kampe  StandardScaler  0.033892  0.982514      10   
4   Gevrey Method (12 features)  StandardScaler  0.039202  0.976607      10   
5                 Random Forest  StandardScaler  0.049162  0.963210       1   
6   Gevrey Method (10 features)  StandardScaler  0.061324  0.942754      10   
7    Gevrey Method (8 features)  StandardScaler  0.070896  0.923489      10   
8                   Correlation  StandardScaler  0.088407  0.881024      10   
9            mRMR (10 features)  StandardScaler  0.088846  0.879840       1   
10           mRMR (14 features)  StandardScaler  0.104790  0.832845      10   

    Best epsilon Best gamma  Features  
0          